# Lab 01 — Linear Regression — Salary vs Experience
**Regression Track** · Beginner · ~40 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Fit simple OLS with sklearn LinearRegression
2. Interpret slope and intercept in business terms
3. Evaluate with R² and residual plots
4. Predict salary at unseen experience values

## Datasets (this folder)
- `Salary_Data.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-01-linear-regression-salary/lab-01-linear-regression-salary.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Salary_Data.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-01-linear-regression-salary"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-01-linear-regression-salary/bundle/dataset.zip"
NEED = ["Salary_Data.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Data Science: Predicting Salary from Experience

> **Scenario:** You are an HR analyst at a mid-size tech company. Leadership wants a defensible model that predicts expected salary from years of experience, so they can benchmark offers and spot pay inequities. You have 30 historical employee records with `YearsExperience` and `Salary`. Your job is to build, evaluate, and interpret a simple linear regression model.
>
> **You will learn:** OLS intuition, sklearn LinearRegression, slope/intercept interpretation, R², residual analysis, train/test split.
> **Time:** ~40 minutes. **Level:** Beginner. **Needs:** pandas + sklearn + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | What it means | Where it shows up |
|---|---|---|
| Linear regression | Fit a straight line `y = mx + b` minimizing squared error | `LinearRegression().fit(X, y)` |
| Slope (m) | Change in salary per extra year of experience | `model.coef_` |
| Intercept (b) | Predicted salary at 0 years experience | `model.intercept_` |
| R² | Fraction of variance explained (1 = perfect) | `model.score(X_test, y_test)` |
| Residuals | Prediction errors (y_true − y_pred) | Residual plot |
| Train/test split | Hold out data to estimate real-world performance | `train_test_split` |

---

## 1. Load and Explore the Data

Linear regression models the relationship between a continuous target and one or more features as a straight line. With a single feature, the model is `Salary = slope × YearsExperience + intercept`. The Ordinary Least Squares (OLS) algorithm finds the line that minimizes the sum of squared vertical distances between the data points and the line.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\Salary_Data.csv")
print(df.shape)
print(df.head())
print(df.describe())
print(df.isnull().sum())


**What to notice:**
- 30 rows, 2 columns — a tiny, clean dataset.
- `YearsExperience` ranges from 1.1 to 10.5 years; `Salary` ranges from ~37k to ~122k.
- No missing values, so we can go straight to modeling.

> **Pitfall:** Always check for missing values before fitting. `LinearRegression` will raise an error on NaNs.

---

## 2. Fit a Linear Regression Model

We split the data into 70% training and 30% testing so we can evaluate the model on unseen data. Then we fit `LinearRegression` on the training set and inspect the learned parameters.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Slope (coef_):     {model.coef_[0]:.2f}")
print(f"Intercept:         {model.intercept_:.2f}")
print(f"R² on train:       {model.score(X_train, y_train):.4f}")
print(f"R² on test:        {model.score(X_test, y_test):.4f}")


**What to notice:**
- The slope tells you the average salary increase per additional year of experience.
- The intercept is the model's salary prediction at 0 years (extrapolation — interpret with caution).
- R² near 0.95 means the line explains ~95% of salary variance — a strong fit.

> **Pitfall:** A high R² on training data can be misleading. Always check test R² to detect overfitting.

---

## 3. Make Predictions

Once fitted, the model can predict salary for any experience value. We predict for 5.5 years and also compute the Mean Squared Error on the test set.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred_5_5 = model.predict([[5.5]])
print(f"Predicted salary at 5.5 years: ${pred_5_5[0]:,.2f}")

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Test MSE:  {mse:,.2f}")
print(f"Test RMSE: {np.sqrt(mse):,.2f}")


**What to notice:**
- The prediction at 5.5 years should be close to the actual salaries of employees with ~5 years of experience.
- RMSE is in the same units as salary (dollars), making it more interpretable than MSE.

> **Pitfall:** `model.predict` expects a 2D array. Use `[[5.5]]`, not `[5.5]`.

---

## 4. Residual Analysis

Residuals are the differences between actual and predicted values. Plotting them helps you check whether the linear model's assumptions hold — residuals should be randomly scattered around zero with no obvious pattern.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_pred, residuals, color="steelblue", edgecolor="k", alpha=0.7)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Predicted Salary")
ax.set_ylabel("Residual (Actual − Predicted)")
ax.set_title("Residual Plot — Salary Prediction")
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f}")
print(f"Residual std:  {residuals.std():.2f}")


**What to notice:**
- Residuals should be roughly symmetric around the red zero line.
- A mean near zero confirms the model is unbiased on average.
- No funnel shape or curve means the linear assumption is reasonable.

> **Pitfall:** A funnel-shaped residual plot (heteroscedasticity) suggests the error variance changes with the prediction — a sign the model may need transformation.

---

## Exercises (do these!)

### Exercise 1 — Predict Salary at 5.5 Years
Use the fitted model to predict the salary for a candidate with 5.5 years of experience.
*Expected: $77,283.39*

**Follow-up:** What is the predicted salary at 0 years? Why might this be unrealistic? Check: intercept value.

### Exercise 2 — Compute Test MSE
Calculate the Mean Squared Error on the held-out test set.
*Expected: MSE = 37,784,662.47 (RMSE = $6,146.92)*

**Follow-up:** Why is RMSE more interpretable than MSE? Check: RMSE value.

### Exercise 3 — Interpret R²
The test R² is approximately 0.95. What does this mean in plain language?
*Expected: The model explains 94.1% of the variance in salary using years of experience.*

**Follow-up:** What would R² = 0 mean? What would R² = 1 mean? Check: R² on train vs test.

<details>
<summary>Hint</summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# Exercise 1
pred = model.predict([[5.5]])
print(pred[0])

# Exercise 2
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, model.predict(X_test))
print(mse)

# Exercise 3
print(model.score(X_test, y_test))
```

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df = pd.read_csv(r"D:\downloads-D\vilabs-images\ailab-vilabs\course-02-classical-machine-learning\datasets\Salary_Data.csv")
X = df[["YearsExperience"]]
y = df["Salary"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# --- Solution 1 ---
pred_5_5 = model.predict([[5.5]])
print(f"Predicted salary at 5.5 years: ${pred_5_5[0]:,.2f}")

# --- Solution 2 ---
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Test MSE:  {mse:,.2f}")
print(f"Test RMSE: {np.sqrt(mse):,.2f}")

# --- Solution 3 ---
r2_train = model.score(X_train, y_train)
r2_test = model.score(X_test, y_test)
print(f"R² train: {r2_train:.4f}")
print(f"R² test:  {r2_test:.4f}")
print(f"The model explains {r2_test*100:.1f}% of salary variance.")

# --- Follow-up 1 ---
print(f"Intercept (salary at 0 years): ${model.intercept_:,.2f}")

# --- Follow-up 2 ---
print(f"RMSE: ${np.sqrt(mse):,.2f} — in salary dollars, unlike MSE.")

# --- Follow-up 3 ---
assert r2_test > 0.90, "R² should be above 0.90"
print(f"R² test = {r2_test:.4f} — strong fit confirmed.")


### What to learn next
- Multiple linear regression with more features
- Regularization (Ridge, Lasso) to prevent overfitting
- Polynomial regression for non-linear relationships

*Files in this folder: Salary_Data.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
